# 04. PyTorch Workflow

Let's explore an example PyTorch end-to-end workflow.

Resources: 
- Ground truth notebook - https://github.com/mrdbourke/pytorch-deep-learning/blob/main/01_pytorch_workflow.ipynb
- Book version of notebook - https://www.learnpytorch.io/01_pytorch_workflow/
- Ask a question - https://github.com/mrdbourke/pytorch-deep-learning/discussions

In [ ]:
what_were_covering = {
    1: "data (prepare and load)",
    2: "build model",
    3: "fitting the model to data (training)",
    4: "making predictions and evaluating a model (inference)",
    5: "saving and loading a model",
    6: "putting it all together"
}
what_were_covering

In [ ]:
import torch
from torch import nn ## nn contains all of PyTorch's building blocks for neural networks
import matplotlib.pyplot as plt

torch.__version__

## 1. Data (preparing and loading)

Data can be anything.... in machine learning

- Excel Spreadsheet
- Images of any kind
- Videos (YouTube has lots of data)
- Audio like songs or podcasts
- DNA
- Text

Machine learning is a game of two parts:
1. Get data into a numerical representation
2. Build a model to learn patterns in that numerical representation.

To showcase this, let's create some known data using the linear regression formula
We'll use a linear regression formula to make a straight line with known parameters.

In [ ]:
# Create *known* parameters
weight = 0.7
bias = 0.3

# Create 
start = 0
end = 1
step = 0.02
X = torch.arange(start, end, step).unsqueeze(dim=1) # add an extra dimension
y = weight * X + bias # linear regression
X[:3], y[:3]

## Splitting data into training and test sets (one of the most important concepts)

Let's create a training and test set with our data

In [ ]:
# Create a train/test split
train_split = int(0.8 * len(X))
X_train, y_train = X[:train_split], y[:train_split]
X_test, y_test = X[train_split:], y[train_split:] 
# we can use skikit learn train test split as it contains random shuffling and splitting functionality

In [ ]:
# Visualizing the data

def plot_predictions(train_data=X_train,
                     train_labels=y_train,
                     test_data=X_test,
                     test_labels=y_test,
                     predictions=None):
    """
    Plots training data, test data and compares predictions.
    """
    plt.figure(figsize=(10, 7))
    
    # Plot training data in blue
    plt.scatter(train_data, train_labels, c="b", s=4, label="Training data")
    
    # Plot test data in green
    plt.scatter(test_data, test_labels, c="g", s=4, label="Testing data")
    
    # Are there predictions?
    if predictions is not None:
        # Plot the predictions if they exist
        plt.scatter(test_data, predictions, c="r", s=4, label="Predictions")
        
    # Show the legend
    plt.legend(prop= {"size": 14})

In [ ]:
plot_predictions()

## 2. Build model

Our first PyTorch model!
This is very exciting... let's do it!

In Linear Regression formula: Y = A + BX, A (bias) and B(weight) are parameters

What our model does:
- Start with random values (weight and bias)
- Look at training data and adjust the random values to better represent (or get closer to) the ideal values (the weight and bias values we used to create a data)

How does it do so?

Through two main algorithms:
1. Gradient Descent (https://www.youtube.com/watch?v=IHZwWFHWa-w)
2. Backpropagation (https://www.youtube.com/watch?v=IHZwWFHWa-w)

In [ ]:
# Create a linear regression model
class LinearRegressionModel(nn.Module): # almost everything in PyTorch inherits from nn.Module (inheriting Base neural network model of PyTorch)
    def __init__(self):
        super().__init__()
        
        self.weights = nn.Parameter(torch.randn(1, # start with random weight and try to adjust it to the ideal weight
            requires_grad=True, # can this parameter be updated via gradient descent?
            dtype=torch.float # PyTorch loves the datatype torch.float32
        ))
        
        self.bias = nn.Parameter(torch.randn(1,
            requires_grad=True,
            dtype=torch.float
        ))
        
    # Forward method to define the computation in the model
    def forward(self, x: torch.Tensor) -> torch.Tensor: # "x" is the input data
        return self.weights * x + self.bias
        
        

## PyTorch model building essentials

- torch.nn - contains all of the building blocks for computational graphs (a neural network can be considered a computational graph)
- torch.nn.Parameter - what parameters should our model try to learn? (weights and bias) (often pytorch layer from torch.nn will set this for us)
- torch.nn.Module - base class for all neural network modules, if you subclass it, you should override forward method
- torch.optim - this is where the optimizers in PyTorch live, they will help with gradient descent
- def forward() - all nn.Module subclasses require you to overwrite forward(), this method defines what happens in the forward 
- torch.utils.data.Dataset
- torch.utils.data.DataLoader
- torchvision.transforms

See more of these essential modules via the PyTorch cheatsheet - https://pytorch.org/tutorials/beginner/ptcheat.html

In [ ]:
torch.randn(1)

## Checking the contents of our PyTorch model

Now we've created a model, let's see what's inside...
So we can check our model parameters or what's inside our model using .parameters().

In [ ]:
# Create a random seed
torch.manual_seed(42)

# Create an instance of the model (this is a subclass of nn.Module)
model_0 = LinearRegressionModel()

# Check out 
list(model_0.parameters())

In [ ]:
# List named parameters
model_0.state_dict()

### Making prediction using `torch.inference_mode()`

To check our model's predictive power, let's see how well it predicts `y_test` based on `X_test`.

When we pass data through our model, it's going to run it through the `forward()` method.

In [ ]:
X_test

In [ ]:
# Make predictions with model
with torch.inference_mode(): # this inference_mode() does not track gradients, also some advantages over no_grad
    y_preds = model_0(X_test)
    
# we can do similiar with no_grad, however inference_mode() is recommended as it has some performance optimizations
with torch.no_grad():
    y_preds = model_0(X_test)
    
y_preds

In [ ]:
plot_predictions(predictions=y_preds)

## 3. Train model

The whole idea of training is for a model to move from some *unknown* parameters (these may be random) to some *known* parameters

Or in the other words from a poor representation of data to a better representation of data

One way to measure how poor or how wrong your models predictions are is to use a loss function.

* Note: Loss function may also be called cost function or criterion in different areas. For our case, we're going to refer to it as a loss function

Things we need to train:

* **Loss Function:** A function to measure how wrong your model's predictions are to the ideal outputs, lower is better 
* **Optimizer:** Takes into account the loss of a model and adjusts the model's parameters (e.g. weight & bias) to improve loss function
* For reading optimizer: https://pytorch.org/docs/stable/optim.html#module-torch.optim

And specifically for PyTorch, we need: 
- A training loop
- A testing loop

In [ ]:
model_0.parameters()

In [ ]:
# Checkout our model's parameters (a parameter is a value that the model sets itself)
model_0.state_dict()

In [ ]:
# Setup a loss function
loss_fn = nn.L1Loss()

# Setup an optimizer (stochastic gradient descent)
optimizer = torch.optim.SGD(params=model_0.parameters(), # what parameters to optimize?
                            lr=0.01) # lr = learning rate = the most important hyperparameter = how much to update the parameters each step

**Q**: Which loss function and optimizer should I use?\
**A**: This will be problem specific. But with experience, you'll get an idea of what works and what doesn't with your problem set

For example, for a regression problem (like ours), a loss function of nn.L1Loss() and optimizer like torch.optim.SGD suffice.\
But for a classification problem like classifying whether a photo is of a dog or a cat, you'll likely want to use a loss function of nn.BCELoss() (binary cross entropy loss)

### Building a training loop (and a testing loop) in PyTorch

A couple of things we need in a training loop:
0. Loop through the data
1. Forward pass (this involves data moving through our model's `forward()` functions)
to make predictions on data - also called forward propagation
2. Calculate the loss (compare forward pass predictions to ground truth labels) 
3. Optimizer zero grad
4. Loss Backward - move backwards through network to calculate the gradients of each of the parameters of our model w.r.t loss (**backpropagation**)
5. Optimizer step - use the optimizer to adjust our model's parameters (**gradient descent**)



In [ ]:
torch.manual_seed(42)

# An epoch is one loop through the data... (this is a hyperparameter we can adjust it ourselves)
epochs = 100

# 0. Loop through the data
for epochs in range(epochs):
    # Set the model to training mode
    model_0.train() #train mode in PyTorch sets all parameters that require gradients to require gradients
      
    # 1. Forward pass on train data using the forward() method inside our model_0
    y_pred = model_0(X_train)
    
    # 2. Calculate the loss
    loss = loss_fn(y_pred, y_train)
    
    # 3. Optimizer zero grad
    optimizer.zero_grad() # start fresh (PyTorch accumulates gradients by default, so if you don't zero them out, they will be added to existing gradients)
    
    # 4. Perform backpropagation on the loss with respect to the parameters of the model
    loss.backward()
    # means compute gradient of every parameter with requires_grad=True
    
    # 5. Step the optimizer (perform gradient descent)
    optimizer.step() # by default how the optimizer changes will accumulate through loop so... we have to zero them above in step 3
    
    # Testing
    model_0.eval() # turns off different settings in the model that are only needed for training (like dropout, batchnorm etc.);
    
    with torch.inference_mode(): # turns off gradient tracking and some memory optimizations
        # 1. Forward pass on test data
        test_pred = model_0(X_test)
        
        # 2. Calculate the loss
        test_loss = loss_fn(test_pred, y_test)
        
    if epochs % 10 == 0:
        print(f"Epoch: {epochs} | Loss: {loss} | Test loss: {test_loss}")

In [ ]:
with torch.inference_mode():
    y_preds_new = model_0(X_test)

In [ ]:
model_0.state_dict()

In [ ]:
weight, bias

In [ ]:
plot_predictions(predictions=y_preds)

In [ ]:
plot_predictions(predictions=y_preds_new)